In [ ]:
#run this cell once in case the environment is not prepared
# !pip install cobra pandas matplotlib matplotlib-venn

import cobra
from cobra.io import read_sbml_model
from cobra.flux_analysis import single_gene_deletion

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2, venn3

import os


## Cell 2 – Loading the SBML Model

In this step, we load the genome-scale metabolic model of *Pseudomonas aeruginosa* from an SBML file.

SBML (Systems Biology Markup Language) is the standard format for exchanging biochemical network models, including:
- Genome-scale metabolic reconstructions  
- Reaction stoichiometry  
- Gene–protein–reaction (GPR) associations  
- Metabolite and reaction annotations  

Using COBRApy, the SBML file is parsed into a `Model` object that can be used for:
- Flux Balance Analysis (FBA)
- Pathway-level flux inspection
- Essential gene and reaction simulations
- Environmental condition manipulation (media, uptake rates)

This cell initializes the core model that will be used throughout all subsequent analyses.

In [ ]:

from cobra.io import read_sbml_model

# Set the path to the SBML model file
MODEL_PATH = "models/MODEL1507180020_url.xml"  # Change this if your file is in a different folder

# Load the genome scale metabolic model
model = read_sbml_model(MODEL_PATH)

# Print a short summary of the loaded model
print(f"Loaded model: {model.id}")
print(f"Number of reactions:   {len(model.reactions)}")
print(f"Number of metabolites: {len(model.metabolites)}")
print(f"Number of genes:       {len(model.genes)}")

## Cell 3 – Identify the Biomass Reaction

If you do not know the exact ID of the biomass reaction, you can search it using a simple string match.  
This cell scans all reactions in the model and returns those whose IDs contain the word “biomass”.>

In [ ]:

from cobra.core import Reaction
from typing import List

# Search for candidate biomass reactions
# This list comprehension checks all reactions and picks those
# whose ID contains the keyword "biomass" (case-insensitive).
biomass_candidates: List[Reaction] = [
    rxn for rxn in model.reactions
    if "biomass" in rxn.id.lower()
]

# Print the list of candidate reactions for manual inspection
biomass_candidates

After finding the reaction ID, set it as objective function

In [ ]:
# replace this ID with the one you got
BIOMASS_RXN_ID = "BIOMASS_PAO" 

biomass_rxn = model.reactions.get_by_id(BIOMASS_RXN_ID)
print(biomass_rxn)
model.objective = biomass_rxn


## Cell 4 - finding the reaction ID for those exchanging carbon and electron acceptors

note: these are the common ones in BIGG models, find the specific carbon sources and acceptors for your model using the code below:

In [5]:
# To find and inspect the exchange reactions use the following code
# for rxn in model.exchanges:
#     print(rxn.id, ":", rxn.name)

EX_CARBON = {
    "glucose":  "EX_glc__D_e",   #adjust to the specific model
    "acetate":  "EX_ac_e",
    "glycerol": "EX_glyc_e",
    "ethanol":  "EX_etoh_e",
}

EX_ACCEPTOR = {
    "oxygen": "EX_o2_e",
    "nitrate": "EX_no3_e",
}

# The byproducts we want to inspect
BYPRODUCTS = {
    "lactate": "EX_lac__L_e",   #edit if needed 
    "acetate": "EX_ac_e",
}

## Cell 5 - Defining the function to reset the environment

In [ ]:
def reset_medium(model, default_uptake=-10.0):
    """
    resets all uptake to zero to add the especific acceptors and sources
    """
    # blocking all exchanges where lower_bound is less than 0, meaning there will be no more uptake
    for rxn in model.exchanges:
        if rxn.lower_bound < 0:
            rxn.lower_bound = 0.0

    # you can also add a set of  fluxes that need to be functioning at all times 
    return model


## Cell 6 - Defining eight deifferent environments

In [11]:
#creating the list of all 8 different conditions
conditions = []

for carbon_name, carbon_ex in EX_CARBON.items():
    for acc_name, acc_ex in EX_ACCEPTOR.items():
        cond_id = f"{carbon_name}_{acc_name}"  # for example: glucose as the source and oxygen as the acceptor
        conditions.append({
            "id": cond_id,
            "carbon_name": carbon_name,
            "carbon_ex": carbon_ex,
            "acceptor_name": acc_name,
            "acceptor_ex": acc_ex,
            "carbon_uptake": -10.0,   # mmol/gDW/h
            "acceptor_uptake": -20.0  # for example: O2 or NO3
        })

conditions


[{'id': 'glucose_oxygen',
  'carbon_name': 'glucose',
  'carbon_ex': 'EX_glc__D_e',
  'acceptor_name': 'oxygen',
  'acceptor_ex': 'EX_o2_e',
  'carbon_uptake': -10.0,
  'acceptor_uptake': -20.0},
 {'id': 'glucose_nitrate',
  'carbon_name': 'glucose',
  'carbon_ex': 'EX_glc__D_e',
  'acceptor_name': 'nitrate',
  'acceptor_ex': 'EX_no3_e',
  'carbon_uptake': -10.0,
  'acceptor_uptake': -20.0},
 {'id': 'acetate_oxygen',
  'carbon_name': 'acetate',
  'carbon_ex': 'EX_ac_e',
  'acceptor_name': 'oxygen',
  'acceptor_ex': 'EX_o2_e',
  'carbon_uptake': -10.0,
  'acceptor_uptake': -20.0},
 {'id': 'acetate_nitrate',
  'carbon_name': 'acetate',
  'carbon_ex': 'EX_ac_e',
  'acceptor_name': 'nitrate',
  'acceptor_ex': 'EX_no3_e',
  'carbon_uptake': -10.0,
  'acceptor_uptake': -20.0},
 {'id': 'glycerol_oxygen',
  'carbon_name': 'glycerol',
  'carbon_ex': 'EX_glyc_e',
  'acceptor_name': 'oxygen',
  'acceptor_ex': 'EX_o2_e',
  'carbon_uptake': -10.0,
  'acceptor_uptake': -20.0},
 {'id': 'glycerol_nitr

## Cell 7 - Running FBA for each condition

In [14]:
def run_fba_for_condition(base_model, condition):
    """
    base_model: cobra.Model
    condition: dict or keys: carbon_ex, acceptor_ex, carbon_uptake, acceptor_uptake
    """
    model = base_model.copy()
    reset_medium(model)

    #  setting uptake for carbon source 
    c_ex = model.reactions.get_by_id(condition["carbon_ex"])
    c_ex.lower_bound = condition["carbon_uptake"]

    # setting uptake for electron acceptor
    e_ex = model.reactions.get_by_id(condition["acceptor_ex"])
    e_ex.lower_bound = condition["acceptor_uptake"]

    solution = model.optimize()
    return model, solution


## Cell 8 - Running FBA for all 8 conditions and creating the data frame for growth results


In [ ]:
results = []

for cond in conditions:
    model_cond, sol = run_fba_for_condition(model, cond)
    growth = sol.objective_value

    # flux of byproducts
    byp_flux = {}
    for name, ex_id in BYPRODUCTS.items():
        try:
            rxn = model_cond.reactions.get_by_id(ex_id)
            byp_flux[name] = sol.fluxes[ex_id]
        except KeyError:
            byp_flux[name] = None

    row = {
        "condition": cond["id"],
        "carbon": cond["carbon_name"],
        "acceptor": cond["acceptor_name"],
        "growth_rate": growth,
    }
    # adding to the rows
    for name, fl in byp_flux.items():
        row[f"flux_{name}"] = fl

    results.append(row)

df_growth = pd.DataFrame(results)
df_growth


## Cell 9 - Growth Rate bar plot for 8 conditions

In [20]:
plt.figure(figsize=(10, 5))
df_growth_sorted = df_growth.sort_values("growth_rate", ascending=False)

plt.bar(df_growth_sorted["condition"], df_growth_sorted["growth_rate"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Biomass flux (1/h)")
plt.title("Growth rate across 8 carbon/electron acceptor conditions")
plt.tight_layout()
plt.show()


NameError: name 'plt' is not defined

## Cell 10 - Choosing the reactions from Glycolysis and TCA pathways
we can do this by either creating a manual list of the reaction IDs or by filtering the subsysytems (if our model has it)

In [ ]:
# Example: if model.reactions has defined subsystems 
# [rxn.id for rxn in model.reactions if rxn.subsystem == "Glycolysis/Gluconeogenesis"]

GLYCOLYSIS_RXNS = [
    # "PFK", "PYK", ...
]

TCA_RXNS = [
    # "CS", "ACONTa", "ACONTb", "ICDHyr", "AKGDH", ...
]

def get_pathway_fluxes(solution, rxn_ids):
    data = {}
    for rid in rxn_ids:
        if rid in solution.fluxes.index:
            data[rid] = solution.fluxes[rid]
    return data


## Cell 11 - extracting and depicting the glycolysis and tca flow for each condition


In [ ]:
pathway_results = []

for cond in conditions:
    model_cond, sol = run_fba_for_condition(model, cond)

    gly_fluxes = get_pathway_fluxes(sol, GLYCOLYSIS_RXNS)
    tca_fluxes = get_pathway_fluxes(sol, TCA_RXNS)

    pathway_results.append({
        "condition": cond["id"],
        "glycolysis": gly_fluxes,
        "tca": tca_fluxes,
    })

# example: viewing the first one
pathway_results[0]


##  example for one pathway and one condition or comparing multiple environments
<div dir=rtl>


In [ ]:
# example: comparing the flow of TCA in multiple environments
selected_conditions = ["glucose_oxygen", "glucose_nitrate"]  # change as  desired 

for cond_name in selected_conditions:
    entry = next(p for p in pathway_results if p["condition"] == cond_name)
    fluxes = entry["tca"]

    plt.figure(figsize=(8, 4))
    plt.bar(fluxes.keys(), fluxes.values())
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Flux (mmol/gDW/h)")
    plt.title(f"TCA fluxes in condition: {cond_name}")
    plt.tight_layout()
    plt.show()

## Cell 12 - inspection of essential genes by single gene deletion

In [ ]:
def gene_essentiality_for_condition(base_model, condition, threshold_ratio=0.01):
    """
    threshold ratio: 0.1, means the genes which the gowth rate after their deletion will be less than 1% of the wild type growth rate
    
    """
    model_cond, sol_wt = run_fba_for_condition(base_model, condition)
    wt_growth = sol_wt.objective_value

    if wt_growth is None or wt_growth <= 1e-9:
        print("No growth in condition", condition["id"])
        return set()

    deletion_res = single_gene_deletion(model_cond, model_cond.genes, processes=1)
    essential_genes = set()

    for idx, row in deletion_res.iterrows():
        gene = row["ids"]
        gr = row["growth"]
        if gr is None:
            continue
        if gr < threshold_ratio * wt_growth:
            essential_genes.add(gene)

    return essential_genes


## Cell 13 - analysing the essential genes for multiple environments and the corrosponding Venn diagram

example for 3 conditions: glucose_oxygen، glucose_nitrate، acetate_oxygen.

In [29]:
# انتخاب چند محیط برای مقایسه
cond_ids_for_venn = ["glucose_oxygen", "glucose_nitrate", "acetate_oxygen"]

ess_sets = {}
for cid in cond_ids_for_venn:
    cond = next(c for c in conditions if c["id"] == cid)
    print("Computing essential genes for:", cid)
    ess_sets[cid] = gene_essentiality_for_condition(model, cond)

ess_sets

Computing essential genes for: glucose_oxygen


NameError: name 'gene_essentiality_for_condition' is not defined

portraying the Venn diagram for 3 conditions

In [ ]:
a, b, c = cond_ids_for_venn
set_a = ess_sets[a]
set_b = ess_sets[b]
set_c = ess_sets[c]

plt.figure(figsize=(6, 6))
venn3([set_a, set_b, set_c], set_labels=(a, b, c))
plt.title("Overlap of essential genes across conditions")
plt.show()

In [ ]:
Note: in case of comparing 2 environments, use Venn2